# 3. Tunable CNN Training
Here we implement a CNN using TensorFlow/Keras to classify LULC patches. We use `keras_tuner` for hyperparameter tuning.


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt
import numpy as np


## Model Definition


In [ ]:
def build_model(hp):
    model = keras.Sequential()
    model.add(keras.Input(shape=(16, 16, 6))) # Assume 6 bands: RGB, NIR, SWIR, NDVI, NDBI
    
    # Tune filters for Conv2D
    filters_1 = hp.Int('filters_1', min_value=32, max_value=128, step=32)
    model.add(layers.Conv2D(filters_1, kernel_size=(3, 3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())
    model.add(layers.MaxPooling2D(pool_size=(2, 2)))

    filters_2 = hp.Int('filters_2', min_value=64, max_value=256, step=64)
    model.add(layers.Conv2D(filters_2, kernel_size=(3, 3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())
    model.add(layers.MaxPooling2D(pool_size=(2, 2)))
    
    model.add(layers.Flatten())
    model.add(layers.Dropout(0.5))
    
    # 11 Classes for ESRI LULC
    model.add(layers.Dense(11, activation='softmax'))
    
    # Tune learning rate
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    directory='my_dir',
    project_name='lulc_tuning'
)
print('Tuner defined successfully.')
